# 05 — Rank-difference deconvolution

Join experimental DMS (if present) with **ESM-2 650M** and SPURS. The KDE model
flags mutants whose evolutionary rank is much worse than their stability rank
(`neg_spurs` or DMS `stability`).

If an `EVmutation` column is present (EVcouplings webserver), it is deconvolved
as well; ESM-2 650M is sufficient to run the example.

## Load inputs

In [1]:
from IPython.display import display
import pandas as pd

from dualde import load_example
from dualde.deconvolution import deconvolve_evo_vs_stability, merge_evo_spurs_dms

EXAMPLE, cfg = load_example()
OUT = EXAMPLE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
PDB_ID = cfg["pdb_id"]

evo_path = OUT / "evolutionary.csv" if (OUT / "evolutionary.csv").is_file() else EXAMPLE / "scores" / "evolutionary.csv"
spurs_path = OUT / f"{PDB_ID}_spurs.csv" if (OUT / f"{PDB_ID}_spurs.csv").is_file() else EXAMPLE / "scores" / f"{PDB_ID}_spurs.csv"

evo = pd.read_csv(evo_path)
spurs = pd.read_csv(spurs_path)
activity = pd.read_csv(EXAMPLE / "dms" / "activity.csv")
stability = pd.read_csv(EXAMPLE / "dms" / "stability.csv")

print("evo", evo.shape, "from", evo_path.name)
print("spurs", spurs.shape)
print("activity", activity.shape, "stability", stability.shape)
display(activity.head(3))
display(stability.head(3))

evo (3444, 10) from evolutionary.csv
spurs (3120, 7)
activity (2935, 3) stability (2923, 3)


,mutant,mutated_sequence,DMS_score
0,L164V,MTASAQPRGRRPGVGVGVVVTSCKHPRCVLLGKRKGSVGAGSFQLP...,1.495371
1,L164R,MTASAQPRGRRPGVGVGVVVTSCKHPRCVLLGKRKGSVGAGSFQLP...,1.228239
2,L164C,MTASAQPRGRRPGVGVGVVVTSCKHPRCVLLGKRKGSVGAGSFQLP...,1.196797


,mutant,mutated_sequence,DMS_score
0,L164V,MTASAQPRGRRPGVGVGVVVTSCKHPRCVLLGKRKGSVGAGSFQLP...,1.031492
1,L164R,MTASAQPRGRRPGVGVGVVVTSCKHPRCVLLGKRKGSVGAGSFQLP...,0.994814
2,L164C,MTASAQPRGRRPGVGVGVVVTSCKHPRCVLLGKRKGSVGAGSFQLP...,0.931596


## Merge

In [2]:
merged = merge_evo_spurs_dms(evo, spurs, activity, stability)
print("merged", merged.shape)
print("columns:", list(merged.columns))
print("missingness (key cols):")
key = [c for c in ["mutant", "activity", "stability", "spurs", "neg_spurs", "ESM2_650M", "EVmutation"] if c in merged.columns]
display(merged[key].isna().mean().to_frame("frac_na"))
display(merged.head())

merged (2844, 19)
columns: ['pos', 'ref', 'aa_to', 'mutant', 'ESM2_35M', 'ESM2_650M', 'ESM2_3B', 'EVmutation2', 'EVmutation', 'EVE', 'activity', 'stability', 'spurs', 'uniprot_position', 'pdb_position', 'wt', 'subs', 'target_pos', 'neg_spurs']
missingness (key cols):


,frac_na
mutant,0.000000
activity,0.000000
stability,0.000000
spurs,0.042194
neg_spurs,0.042194
ESM2_650M,0.000000
EVmutation,0.214838


,pos,ref,aa_to,mutant,ESM2_35M,ESM2_650M,ESM2_3B,EVmutation2,EVmutation,EVE,activity,stability,spurs,uniprot_position,pdb_position,wt,subs,target_pos,neg_spurs
0,2,T,A,T2A,0.326670,3.072813,1.975866,-0.070448,NaN,NaN,1.008568,1.291103,NaN,NaN,NaN,T,A,2,NaN
1,2,T,D,T2D,-0.681067,-1.362573,-2.012577,0.010724,NaN,NaN,0.589723,1.271988,NaN,NaN,NaN,T,D,2,NaN
2,2,T,E,T2E,-0.218684,-0.422243,-1.359274,-0.469386,NaN,NaN,0.753017,1.233425,NaN,NaN,NaN,T,E,2,NaN
3,2,T,F,T2F,-2.482751,-3.590522,-4.804127,-2.532630,NaN,NaN,1.116319,1.224012,NaN,NaN,NaN,T,F,2,NaN
4,2,T,G,T2G,-0.968067,-0.945656,-1.915704,-0.613750,NaN,NaN,1.158185,0.371464,NaN,NaN,NaN,T,G,2,NaN


## KDE models (evolution vs stability proxies)

In [3]:
evo_cols = [c for c in ("ESM2_650M", "EVmutation") if c in merged.columns]
print("deconvolving:", evo_cols, "vs neg_spurs and DMS stability")
merged = deconvolve_evo_vs_stability(merged, evo_cols=evo_cols, stability_cols=("neg_spurs", "stability"))
model_cols = [c for c in merged.columns if str(c).endswith("_model")]
print("model columns:", model_cols)
merged.to_csv(OUT / "deconvolution.csv", index=False)
print("wrote", OUT / "deconvolution.csv")

deconvolving: ['ESM2_650M', 'EVmutation'] vs neg_spurs and DMS stability


model columns: ['ESM2_650M_neg_spurs_model', 'ESM2_650M_stability_model', 'EVmutation_neg_spurs_model', 'EVmutation_stability_model']
wrote /lustre/scratch125/gengen/projects_v2/sgb_enzyme_design/evedesign_prelim/dualde/examples/NUDT15_HUMAN/outputs/deconvolution.csv


Higher `*_model` = more likely an evolutionary outlier given the stability score (putative catalytic / functional effect).

In [4]:
score_col = cfg.get("score_col", "ESM2_650M_neg_spurs_model")
if score_col not in merged.columns:
    score_col = model_cols[0]
print("inspecting", score_col)
print(merged[score_col].describe())
cols = [c for c in ["mutant", "target_pos", "activity", "stability", "neg_spurs", score_col] if c in merged.columns]
display(merged.nlargest(8, score_col)[cols])

inspecting ESM2_650M_neg_spurs_model
count    2724.000000
mean        0.491033
std         0.290456
min         0.000379
25%         0.243009
50%         0.494511
75%         0.740720
max         1.000000
Name: ESM2_650M_neg_spurs_model, dtype: float64


,mutant,target_pos,activity,stability,neg_spurs,ESM2_650M_neg_spurs_model
2137,W125L,125,0.324228,0.027835,-2.842905,1.000000
398,H25D,25,0.605674,-0.006666,-2.771365,0.999364
2311,L134T,134,0.929546,0.551787,-2.012198,0.998267
2289,Q133P,133,0.471691,0.196806,-2.538630,0.997957
2551,P148G,148,0.441097,0.082670,-2.556811,0.996949
407,H25P,25,0.312222,0.033769,-3.314392,0.996867
2572,F149K,149,0.538652,0.729972,-2.205232,0.995730
2145,W125V,125,0.462671,0.057317,-2.726957,0.995307
